In [19]:
import pandas as pd
import numpy as np
import os

ModuleNotFoundError: No module named 'pandas'

In [38]:
# Define the folder path and the column names
folder_path = '/Users/jul/Desktop/uni/Data Analytics/project/PROBE-202411'
columns = [
    'VehicleID',
    'gpsvalid',
    'lat',
    'lon',
    'timestamp',
    'speed',
    'heading',
    'for_hire_light',
    'engine_acc'
]

In [39]:
# Initialize an empty list to store the dataframes
all_dfs = []

In [40]:
# Loop through all files in the directory
for filename in os.listdir(folder_path):
    # Check if the file is a CSV file
    if filename.endswith('.csv.out'):
        file_path = os.path.join(folder_path, filename)

        # Read the CSV file into a dataframe with the specified column names
        df = pd.read_csv(file_path, names=columns)

        # Append the dataframe to the list
        all_dfs.append(df)

# Concatenate all dataframes in the list into a single dataframe
combined_df = pd.concat(all_dfs, ignore_index=True)

In [41]:
combined_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc
0,t7K8v5g8YiXmnbuvfVH4t4qZydQ,1,13.85360,100.55141,2024-11-25 23:54:56,0,63,1,0
1,ySQq59oNQ+kk5G1bx796QOjUQ7M,1,13.71034,100.60201,2024-11-25 23:55:42,36,59,1,1
2,pfmJE+AxHlNbR6Nxhn9t3sDGx7k,1,13.84241,100.57710,2024-11-25 23:56:00,76,29,1,1
3,mz/5Q2opqjHf4kn5WZHoWuBB7VE,1,13.83726,100.60883,2024-11-25 23:55:35,0,285,0,0
4,6dvZqj2Qoq3FvsgF1dDiK827RjM,1,13.82041,100.57500,2024-11-25 23:56:47,61,342,0,1
...,...,...,...,...,...,...,...,...,...
55629589,J5DxZvsj/r9b6Kex8b2OLREv7/o,1,7.87370,98.29954,2024-11-23 23:59:47,0,52,0,0
55629590,QAbyFmQazKLgfGUkfUAPX3HCEIo,1,13.83596,100.64203,2024-11-23 23:59:46,0,64,0,1
55629591,I5AeppcV2mKC7aNKFtyf7WX7QR4,1,7.83091,98.34943,2024-11-23 23:59:47,0,187,0,0
55629592,R1hDu+U6GKLvdQVVVkVp5X/ChdE,1,14.77243,101.52135,2024-11-23 23:59:47,62,195,0,1


In [42]:
cleaned_df = combined_df.dropna()


In [43]:
cleaned_taxi_df = cleaned_df[
    (combined_df['gpsvalid'] == 1) &
    (combined_df['engine_acc'] == 1) &
    (combined_df['lat'].between(-90, 90)) &
    (combined_df['lon'].between(-180, 180)) &
    (combined_df['speed'] >= 0)
].copy().reset_index(drop=True)

In [44]:
# Find the unique VehicleIDs of every vehicle that EVER reported for_hire_light = 1
# This is our most reliable definition of a taxi.
taxi_ids = cleaned_taxi_df[cleaned_taxi_df['for_hire_light'] == 1]['VehicleID'].unique()

In [45]:
# Now, filter the main dataframe to keep ALL records for these identified taxis
# This gives us their full journey, not just when their light was on.
taxis_df = cleaned_taxi_df[cleaned_taxi_df['VehicleID'].isin(taxi_ids)].copy()

In [46]:
# The 'timestamp' column is just text right now. We need to convert it.
taxis_df['timestamp'] = pd.to_datetime(taxis_df['timestamp'])

Sort Data

In [47]:
# Sort by the vehicle first, then by the time.
taxis_df.sort_values(by=['VehicleID', 'timestamp'], inplace=True)

In [48]:
# Reset the index after sorting for a clean DataFrame
taxis_df.reset_index(drop=True, inplace=True)

In [49]:
# Extract time-based features from the timestamp
taxis_df['hour'] = taxis_df['timestamp'].dt.hour
taxis_df['day_of_week'] = taxis_df['timestamp'].dt.dayofweek # Monday=0, Sunday=6
taxis_df['is_weekend'] = (taxis_df['day_of_week'] >= 5).astype(int)

In [50]:
# --- Advanced: Identify Trips ---
# A trip starts when the 'for_hire_light' changes from 1 (empty) to 0 (occupied).
# We can create a 'trip_id' for each taxi.

# First, detect the change from 1 to 0
taxis_df['trip_start'] = (taxis_df['for_hire_light'].shift(1) == 1) & (taxis_df['for_hire_light'] == 0)

In [51]:
# We also need to ensure it's the same vehicle
taxis_df['trip_start'] = taxis_df['trip_start'] & (taxis_df['VehicleID'].shift(1) == taxis_df['VehicleID'])

In [52]:
# Now create a unique ID for each trip using a cumulative sum
taxis_df['trip_id'] = taxis_df.groupby('VehicleID')['trip_start'].cumsum()

In [53]:
# Let's clean up the intermediate column
taxis_df.drop(columns=['trip_start'], inplace=True)

In [54]:
taxis_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0


In [55]:
# cars with trip id
taxis_df.groupby('VehicleID')["trip_id"].nunique()

VehicleID
++qQzutWwL31NcUo8s0jiGZzzS0      1
+1indEOKr/ikPVrJQTVjw4FGxBE      1
+20prWr63K5svsMtTmdqLnmsTGE    294
+A3arvBS15eOvdHE+E06+Ng28+E    245
+BAgYWCbvz0z377ef3Yp687Pp+0    364
                              ... 
zt6r6X6cjBVDqWxKcbyXkn1Cydw    255
zvkvMBxj2VDpNjIfmifwwxl6nMo     57
zwNAI8pHuCgPOF5NMs2nVaXBG04    298
zyy0Hv5yMoClNPQlx1tR4NBssFI      1
zzLYPcDONaA8lLF2aJYFKnoRDQ4      1
Name: trip_id, Length: 2016, dtype: int64

In [56]:
#how many taxis extracted
taxis_df['VehicleID'].nunique()

2016

In [57]:
#how many trips extracted
taxis_df.groupby('VehicleID')['trip_id'].max().sum()

np.int64(419447)

Calculate Idle Time

In [58]:
# It creates the unique ID for each continuous idle period.
taxis_df['idle_start'] = (taxis_df['for_hire_light'].shift(1) == 0) & \
                         (taxis_df['for_hire_light'] == 1) & \
                         (taxis_df['VehicleID'].shift(1) == taxis_df['VehicleID'])
taxis_df['idle_period_id'] = taxis_df.groupby('VehicleID')['idle_start'].cumsum()
taxis_df.loc[taxis_df['trip_id'] == 0, 'idle_period_id'] = 0

In [59]:
# Create a temporary DataFrame with only the idle data points
idle_periods_df = taxis_df[taxis_df['for_hire_light'] == 1].copy()

# Group by each unique idle period to calculate its duration
idle_summary = idle_periods_df.groupby(['VehicleID', 'idle_period_id']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max')
).reset_index()

# Calculate the duration in minutes
idle_summary['idle_duration_minutes'] = (idle_summary['end_time'] - idle_summary['start_time']).dt.total_seconds() / 60 + 1

print("--- This is the information we will merge back ---")
display(idle_summary[['VehicleID', 'idle_period_id', 'idle_duration_minutes']].head())

--- This is the information we will merge back ---


,VehicleID,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,0,42429.666667
1,+1indEOKr/ikPVrJQTVjw4FGxBE,0,42957.700000
2,+20prWr63K5svsMtTmdqLnmsTGE,0,29.000000
3,+20prWr63K5svsMtTmdqLnmsTGE,2,22.000000
4,+20prWr63K5svsMtTmdqLnmsTGE,3,32.000000


In [60]:
# We use a 'left' merge to ensure we keep ALL original rows from taxis_df
# The merge will add the 'idle_duration_minutes' from the summary to the main table
# based on the matching VehicleID and idle_period_id.
taxis_df = pd.merge(
    taxis_df,
    idle_summary[['VehicleID', 'idle_period_id', 'idle_duration_minutes']],
    on=['VehicleID', 'idle_period_id'],
    how='left'
)

# After merging, the rows that were NOT idle (i.e., part of a trip) will have NaN
# for the idle duration. We should fill these with 0.
taxis_df['idle_duration_minutes'].fillna(0, inplace=True)

/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_96180/1096343678.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxis_df['idle_duration_minutes'].fillna(0, inplace=True)


In [61]:
taxis_df.drop(columns=['idle_start'], inplace=True)

In [62]:
taxis_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0,0,42429.666667
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0,0,42429.666667
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0,0,42429.666667
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0,0,42429.666667
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0,0,42429.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0,0,42573.866667
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0,0,42573.866667
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0,0,42573.866667
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0,0,42573.866667


Filter out irrelavant date and time

In [63]:
unique_years = taxis_df['timestamp'].dt.year.unique()
unique_years.sort()
unique_years

array([1970, 2024], dtype=int32)

In [64]:
# filter out years more than 2024 and less than 2015
year_series = taxis_df['timestamp'].dt.year

# Keep only the rows where the year is between 2016 and 2024 (inclusive)
# "later than 2015" means >= 2016
# "drop further than 2024" means <= 2024

taxis_df = taxis_df[(year_series == 2024)].copy()

In [65]:
taxis_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0,0,42429.666667
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0,0,42429.666667
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0,0,42429.666667
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0,0,42429.666667
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0,0,42429.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0,0,42573.866667
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0,0,42573.866667
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0,0,42573.866667
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0,0,42573.866667


In [ ]:
taxis_df.groupby('trip_id')["VehicleID"].nunique()

Filter to Just BMR Bangkok

In [67]:
#Define Regions
BKK_REGION_BOUNDS = {
    'min_lat': 13.4,
    'max_lat': 14.2,
    'min_lon': 99.9,
    'max_lon': 101.3
}

In [68]:
taxis_df_bkk = taxis_df[
    (taxis_df['lat'] >= BKK_REGION_BOUNDS['min_lat']) &
    (taxis_df['lat'] <= BKK_REGION_BOUNDS['max_lat']) &
    (taxis_df['lon'] >= BKK_REGION_BOUNDS['min_lon']) &
    (taxis_df['lon'] <= BKK_REGION_BOUNDS['max_lon'])
].copy()

In [69]:
taxis_df_bkk

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0,0,42429.666667
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0,0,42429.666667
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0,0,42429.666667
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0,0,42429.666667
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0,0,42429.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0,0,42573.866667
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0,0,42573.866667
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0,0,42573.866667
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0,0,42573.866667


In [70]:
#filter speed
taxis_df_bkk = taxis_df_bkk[taxis_df_bkk['speed'] <= 180].copy()

In [71]:
taxis_df_bkk

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0,0,42429.666667
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0,0,42429.666667
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0,0,42429.666667
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0,0,42429.666667
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0,0,42429.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0,0,42573.866667
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0,0,42573.866667
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0,0,42573.866667
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0,0,42573.866667


In [73]:
#how many taxis extracted
taxis_df_bkk['VehicleID'].nunique()

1934

In [74]:
taxis_df_bkk.groupby('VehicleID')['trip_id'].max().sum()

np.int64(418401)